In [17]:
!pip install nibabel scikit-image batchgenerators
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
from skimage.transform import resize
import os
from glob import glob

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [18]:
import numpy as np
from sklearn.model_selection import train_test_split

class BratsDataset(Dataset):
    def __init__(self, data_dir, case_dirs, mode='train', patch_size=(128, 128, 128), transform=None):
        """
        Args:
            data_dir: 数据根目录
            case_dirs: 病例目录列表
            mode: 'train', 'val'
            patch_size: 训练时使用的patch尺寸
            transform: 数据增强
        """
        self.data_dir = data_dir
        self.case_dirs = case_dirs
        self.mode = mode
        self.patch_size = patch_size
        self.transform = transform

    def _get_case_dirs(self):
        """获取所有病例的目录路径"""
        pattern = os.path.join(self.data_dir, "BraTS*")
        case_dirs = glob(pattern)
        return [d for d in case_dirs if os.path.isdir(d)]

    # 其他方法保持不变...
    def _load_nifti(self, file_path):
        """加载NIfTI文件并返回numpy数组"""
        img = nib.load(file_path)
        data = img.get_fdata()
        return data, img.affine

    def _normalize(self, data):
        """标准化图像数据"""
        data = np.clip(data, np.percentile(data, 0.5), np.percentile(data, 99.5))
        data = (data - data.mean()) / (data.std() + 1e-8)
        return data

    def _find_modality_files(self, case_dir):
        """查找病例的4个模态文件和分割文件"""
        files = {}

        modalities = {
            't1': '*t1.nii.gz',
            't1ce': '*t1ce.nii.gz',
            't2': '*t2.nii.gz',
            'flair': '*flair.nii.gz',
            'seg': '*seg.nii.gz'  # 分割标签
        }

        for mod, pattern in modalities.items():
            mod_files = glob(os.path.join(case_dir, pattern))
            if mod_files:
                files[mod] = mod_files[0]
            else:
                # 如果是验证模式且缺少分割文件，可以跳过
                if mod == 'seg' and self.mode == 'val':
                    continue
                else:
                    raise FileNotFoundError(f"在 {case_dir} 中找不到 {mod} 文件")

        return files

    def __len__(self):
        return len(self.case_dirs)

    def __getitem__(self, idx):
        case_dir = self.case_dirs[idx]
        files = self._find_modality_files(case_dir)

        # 加载所有模态
        modalities_data = []
        for mod in ['t1', 't1ce', 't2', 'flair']:
            data, affine = self._load_nifti(files[mod])
            data = self._normalize(data)
            modalities_data.append(data)

        # 堆叠模态 (H, W, D, 4)
        image = np.stack(modalities_data, axis=-1)

        # 加载分割标签
        seg_data = np.zeros_like(image[..., 0])  # 默认全零
        if 'seg' in files:
            seg_data, _ = self._load_nifti(files['seg'])
            seg_data = self._remap_seg_labels(seg_data)

        # 调整尺寸到目标patch size
        if image.shape[:3] != self.patch_size:
            image = self._resize_volume(image, self.patch_size)
            seg_data = self._resize_volume(seg_data, self.patch_size, is_seg=True)

        # 转换为PyTorch格式: (C, D, H, W)
        image = torch.from_numpy(image).permute(3, 2, 0, 1).float()  # (4, D, H, W)
        seg_data = torch.from_numpy(seg_data).permute(2, 0, 1).long()  # (D, H, W)

        # 数据增强（仅训练集）
        if self.transform and self.mode == 'train':
            image, seg_data = self.transform(image, seg_data)

        return image, seg_data

    def _remap_seg_labels(self, seg_data):
        """重新映射BraTS标签为连续整数"""
        remapped_seg = np.zeros_like(seg_data)
        remapped_seg[seg_data == 1] = 1  # 坏死
        remapped_seg[seg_data == 2] = 2  # 水肿
        remapped_seg[seg_data == 4] = 3  # 增强肿瘤
        return remapped_seg

    def _resize_volume(self, volume, target_shape, is_seg=False):
        """调整3D体积尺寸"""
        if is_seg:
            order = 0  # 最近邻插值
        else:
            order = 1  # 线性插值

        resized_slices = []
        for i in range(volume.shape[2]):
            if volume.ndim == 4:  # 多模态图像
                slice_resized = resize(volume[:, :, i, :], target_shape[:2], order=order, preserve_range=True)
            else:  # 分割标签
                slice_resized = resize(volume[:, :, i], target_shape[:2], order=order, preserve_range=True)
            resized_slices.append(slice_resized)

        resized_volume = np.stack(resized_slices, axis=2)

        # 调整深度维度
        if resized_volume.shape[2] != target_shape[2]:
            depth_resized = []
            for i in range(resized_volume.shape[0]):
                if volume.ndim == 4:
                    depth_slice = resize(resized_volume[i, :, :, :], (target_shape[1], target_shape[2]), order=order, preserve_range=True)
                else:
                    depth_slice = resize(resized_volume[i, :, :], (target_shape[1], target_shape[2]), order=order, preserve_range=True)
                depth_resized.append(depth_slice)
            resized_volume = np.stack(depth_resized, axis=0)

        return resized_volume

In [19]:
def create_data_loaders_single_dir(data_dir, batch_size=1, patch_size=(128, 128, 128), val_ratio=0.2):
    """从单一训练目录创建训练和验证数据加载器"""

    # 获取所有病例
    pattern = os.path.join(data_dir, "BraTS*")
    all_case_dirs = [d for d in glob(pattern) if os.path.isdir(d)]

    print(f"找到 {len(all_case_dirs)} 个病例")

    # 分割训练集和验证集
    train_dirs, val_dirs = train_test_split(
        all_case_dirs,
        test_size=val_ratio,
        random_state=42  # 固定随机种子确保可重复性
    )

    print(f"训练集: {len(train_dirs)} 个病例")
    print(f"验证集: {len(val_dirs)} 个病例")

    # 数据集
    train_dataset = BratsDataset(
        data_dir=data_dir,
        case_dirs=train_dirs,
        mode='train',
        patch_size=patch_size,
        transform=BratsTransform(p=0.5)
    )

    val_dataset = BratsDataset(
        data_dir=data_dir,
        case_dirs=val_dirs,
        mode='val',
        patch_size=patch_size,
        transform=None
    )

    # 数据加载器
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0
    )

    return train_loader, val_loader

In [20]:
from google.colab import drive

drive.mount('/content/drive')
def test_your_data():
    DATA_PATH = "/content/drive/MyDrive/Biomedical/MICCAI_BraTS2020_TrainingData"

    print("正在测试你的数据...")

    case_pattern = os.path.join(DATA_PATH, "BraTS*")
    case_dirs = glob(case_pattern)
    case_dirs = [d for d in case_dirs if os.path.isdir(d)]

    print(f"找到 {len(case_dirs)} 个病例目录")

    # 检查第一个病例的文件
    first_case = case_dirs[0]
    case_name = os.path.basename(first_case)
    print(f"\n检查病例: {case_name}")

    expected_files = {
        't1': os.path.join(first_case, f"{case_name}_t1.nii"),
        't1ce': os.path.join(first_case, f"{case_name}_t1ce.nii"),
        't2': os.path.join(first_case, f"{case_name}_t2.nii"),
        'flair': os.path.join(first_case, f"{case_name}_flair.nii"),
        'seg': os.path.join(first_case, f"{case_name}_seg.nii")
    }
    print("查找文件:")
    for mod, file_path in expected_files.items():
        exists = os.path.exists(file_path)
        status = "✓" if exists else "✗"
        print(f"  {status} {os.path.basename(file_path)}: {'找到' if exists else '未找到'}")

    # 检查所有必要文件是否存在
    required_files = ['t1', 't1ce', 't2', 'flair', 'seg']
    missing_files = [mod for mod in required_files if not os.path.exists(expected_files[mod])]

    if missing_files:
        print(f"\n缺少文件: {missing_files}")
        # 显示实际存在的文件
        print(f"\n病例目录中的实际文件:")
        for file in os.listdir(first_case):
            print(f"  {file}")
        return False

    print(f"\n✓ 所有文件都已找到!")

    # 测试直接加载一个文件
    print("\n测试直接文件加载...")
    try:
        # 测试加载一个模态
        test_file = expected_files['t1']
        img = nib.load(test_file)
        data = img.get_fdata()
        print(f"✓ 文件加载成功!")
        print(f"  数据维度: {data.shape}")
        print(f"  数据值范围: [{data.min():.1f}, {data.max():.1f}]")
        print(f"  数据类型: {data.dtype}")

        return True

    except Exception as e:
        print(f"文件加载失败: {e}")
        return False

# 运行测试
if __name__ == "__main__":
    test_your_data()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
正在测试你的数据...
找到 369 个病例目录

检查病例: BraTS20_Training_360
查找文件:
  ✓ BraTS20_Training_360_t1.nii: 找到
  ✓ BraTS20_Training_360_t1ce.nii: 找到
  ✓ BraTS20_Training_360_t2.nii: 找到
  ✓ BraTS20_Training_360_flair.nii: 找到
  ✓ BraTS20_Training_360_seg.nii: 找到

✓ 所有文件都已找到!

测试直接文件加载...
✓ 文件加载成功!
  数据维度: (240, 240, 155)
  数据值范围: [0.0, 536.7]
  数据类型: float64


In [22]:
class BratsDataset(Dataset):
    def __init__(self, data_dir, case_dirs, mode='train', patch_size=(128, 128, 128), transform=None):
        self.data_dir = data_dir
        self.case_dirs = case_dirs
        self.mode = mode
        self.patch_size = patch_size
        self.transform = transform

    def _load_nifti(self, file_path):
        """加载.nii文件"""
        try:
            img = nib.load(file_path)
            data = img.get_fdata()
            return data, img.affine
        except Exception as e:
            print(f"加载文件失败 {file_path}: {e}")
            raise

    def _normalize(self, data):
        """标准化图像数据"""
        data = np.clip(data, np.percentile(data, 0.5), np.percentile(data, 99.5))
        data = (data - data.mean()) / (data.std() + 1e-8)
        return data

    def _find_modality_files(self, case_dir):
        """查找病例的4个模态文件和分割文件"""
        case_name = os.path.basename(case_dir)
        files = {}

        modalities = {
            't1': f"{case_name}_t1.nii",
            't1ce': f"{case_name}_t1ce.nii",
            't2': f"{case_name}_t2.nii",
            'flair': f"{case_name}_flair.nii",
            'seg': f"{case_name}_seg.nii"
        }

        for mod, filename in modalities.items():
            file_path = os.path.join(case_dir, filename)
            if os.path.exists(file_path):
                files[mod] = file_path
            else:
                # 修复错误信息：显示文件名而不是模态名
                if mod == 'seg' and self.mode == 'val':
                    continue  # 验证集可以没有分割标签
                else:
                    raise FileNotFoundError(f"在 {case_dir} 中找不到文件: {filename}")

        return files

    def __len__(self):
        return len(self.case_dirs)

    def __getitem__(self, idx):
        case_dir = self.case_dirs[idx]
        files = self._find_modality_files(case_dir)

        # 加载所有模态
        modalities_data = []
        for mod in ['t1', 't1ce', 't2', 'flair']:
            data, affine = self._load_nifti(files[mod])
            data = self._normalize(data)
            modalities_data.append(data)

        # 堆叠模态 (H, W, D, 4)
        image = np.stack(modalities_data, axis=-1)

        # 加载分割标签
        seg_data, _ = self._load_nifti(files['seg'])
        seg_data = self._remap_seg_labels(seg_data)

        # 调整尺寸到目标patch size
        if image.shape[:3] != self.patch_size:
            image = self._resize_volume(image, self.patch_size)
            seg_data = self._resize_volume(seg_data, self.patch_size, is_seg=True)

        # 转换为PyTorch格式: (C, D, H, W)
        image = torch.from_numpy(image).permute(3, 2, 0, 1).float()  # (4, D, H, W)
        seg_data = torch.from_numpy(seg_data).permute(2, 0, 1).long()  # (D, H, W)

        # 数据增强（仅训练集）
        if self.transform and self.mode == 'train':
            image, seg_data = self.transform(image, seg_data)

        return image, seg_data

    def _remap_seg_labels(self, seg_data):
        """重新映射BraTS标签为连续整数"""
        remapped_seg = np.zeros_like(seg_data, dtype=np.int64)
        remapped_seg[seg_data == 1] = 1  # 坏死
        remapped_seg[seg_data == 2] = 2  # 水肿
        remapped_seg[seg_data == 4] = 3  # 增强肿瘤
        return remapped_seg

    def _resize_volume(self, volume, target_shape, is_seg=False):
        """调整3D体积尺寸"""
        if is_seg:
            order = 0  # 最近邻插值
        else:
            order = 1  # 线性插值

        # 创建目标形状（保持通道数）
        if volume.ndim == 4:  # 多模态图像
            target_shape_with_channels = target_shape + (volume.shape[3],)
        else:  # 分割标签
            target_shape_with_channels = target_shape

        # 使用skimage.resize调整尺寸
        resized_volume = resize(
            volume,
            target_shape_with_channels,
            order=order,
            preserve_range=True,
            anti_aliasing=not is_seg  # 分割标签不需要抗锯齿
        )

        return resized_volume

# 重新测试修复后的数据集
def test_fixed_dataset():
    DATA_PATH = "/content/drive/MyDrive/Biomedical/MICCAI_BraTS2020_TrainingData"

    print("测试修复后的数据集...")

    # 获取所有病例
    case_dirs = [d for d in glob(os.path.join(DATA_PATH, "BraTS*")) if os.path.isdir(d)]
    print(f"找到 {len(case_dirs)} 个病例")

    # 使用前2个病例进行测试
    test_cases = case_dirs[:2]

    try:
        test_dataset = BratsDataset(
            data_dir=DATA_PATH,
            case_dirs=test_cases,
            mode='train',
            patch_size=(128, 128, 128),
            transform=None
        )

        print("✓ 数据集创建成功!")
        print(f"  数据集大小: {len(test_dataset)} 个样本")

        # 测试所有样本
        for i in range(len(test_dataset)):
            image, seg = test_dataset[i]
            print(f"\n样本 {i}:")
            print(f"  图像维度: {image.shape}")
            print(f"  标签维度: {seg.shape}")
            print(f"  图像值范围: [{image.min():.3f}, {image.max():.3f}]")
            print(f"  标签类别: {torch.unique(seg)}")

        return True

    except Exception as e:
        print(f"数据集测试失败: {e}")
        import traceback
        traceback.print_exc()
        return False

# 运行修复后的测试
test_fixed_dataset()

测试修复后的数据集...
找到 369 个病例
✓ 数据集创建成功!
  数据集大小: 2 个样本

样本 0:
  图像维度: torch.Size([4, 128, 128, 128])
  标签维度: torch.Size([128, 128, 128])
  图像值范围: [-0.337, 5.174]
  标签类别: tensor([0, 1, 2, 3])

样本 1:
  图像维度: torch.Size([4, 128, 128, 128])
  标签维度: torch.Size([128, 128, 128])
  图像值范围: [-0.489, 4.316]
  标签类别: tensor([0, 1, 2, 3])


True

In [26]:
import random
import torchvision.transforms.functional as F

class BratsTransform:
    """BraTS数据增强类"""
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, image, seg):
        """
        Args:
            image: (4, D, H, W) 4模态图像
            seg: (D, H, W) 分割标签
        Returns:
            增强后的图像和标签
        """
        # 随机翻转
        if random.random() < self.p:
            # 随机选择翻转的轴
            axes = [1, 2, 3]  # D, H, W 轴
            flip_axis = random.choice(axes)
            image = torch.flip(image, [flip_axis])
            seg = torch.flip(seg, [flip_axis-1])  # seg没有通道维度

        # 随机旋转 (90°, 180°, 270°)
        if random.random() < self.p:
            k = random.randint(1, 3)  # 旋转次数
            # 在H,W平面旋转
            image = torch.rot90(image, k, dims=[2, 3])
            seg = torch.rot90(seg, k, dims=[1, 2])

        # 随机亮度调整（只对图像，不对标签）
        if random.random() < self.p:
            brightness_factor = random.uniform(0.8, 1.2)
            image = image * brightness_factor

        # 随机高斯噪声
        if random.random() < self.p:
            noise_std = random.uniform(0.0, 0.1)
            noise = torch.randn_like(image) * noise_std
            image = image + noise

        return image, seg

# 现在重新创建数据加载器（使用完整的数据增强）
def create_brats_data_loaders():
    """创建BraTS数据加载器"""
    DATA_PATH = "/content/drive/MyDrive/Biomedical/MICCAI_BraTS2020_TrainingData"

    print("创建数据加载器...")

    # 获取所有病例
    case_dirs = [d for d in glob(os.path.join(DATA_PATH, "BraTS*")) if os.path.isdir(d)]
    print(f"总病例数: {len(case_dirs)}")

    # 分割训练集和验证集 (80% 训练, 20% 验证)
    train_dirs, val_dirs = train_test_split(
        case_dirs,
        test_size=0.2,
        random_state=42,
        shuffle=True
    )

    print(f"训练集: {len(train_dirs)} 个病例")
    print(f"验证集: {len(val_dirs)} 个病例")

    # 创建数据集 - 现在BratsTransform已定义
    train_dataset = BratsDataset(
        data_dir=DATA_PATH,
        case_dirs=train_dirs,
        mode='train',
        patch_size=(128, 128, 128),
        transform=BratsTransform(p=0.5)  # 50%概率应用每种增强
    )

    val_dataset = BratsDataset(
        data_dir=DATA_PATH,
        case_dirs=val_dirs,
        mode='val',
        patch_size=(128, 128, 128),
        transform=None  # 验证集不用数据增强
    )

    # 创建数据加载器
    train_loader = DataLoader(
        train_dataset,
        batch_size=1,  # 3D数据batch_size通常为1
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    print("✓ 数据加载器创建成功!")
    return train_loader, val_loader

In [27]:
def test_data_augmentation():
    """测试数据增强效果"""
    DATA_PATH = "/content/drive/MyDrive/Biomedical/MICCAI_BraTS2020_TrainingData"

    case_dirs = [d for d in glob(os.path.join(DATA_PATH, "BraTS*")) if os.path.isdir(d)]
    test_case = case_dirs[0]

    # 创建有数据增强的数据集
    aug_dataset = BratsDataset(
        data_dir=DATA_PATH,
        case_dirs=[test_case],
        mode='train',
        patch_size=(64, 64, 64),
        transform=BratsTransform(p=1.0)  # 100%概率应用增强，便于测试
    )

    print("测试数据增强...")
    original_image, original_seg = aug_dataset[0]  # 会应用增强

    print(f"增强后图像维度: {original_image.shape}")
    print(f"增强后标签维度: {original_seg.shape}")
    print(f"图像值范围: [{original_image.min():.3f}, {original_image.max():.3f}]")
    print(f"标签类别: {torch.unique(original_seg)}")

    # 测试多次增强，观察变化
    print("\n测试多次增强的随机性:")
    for i in range(3):
        aug_image, aug_seg = aug_dataset[0]  # 每次都会随机增强
        print(f"  第{i+1}次 - 图像均值: {aug_image.mean():.3f}, 标签类别: {torch.unique(aug_seg)}")

# 测试数据增强
test_data_augmentation()

测试数据增强...
增强后图像维度: torch.Size([4, 64, 64, 64])
增强后标签维度: torch.Size([64, 64, 64])
图像值范围: [-0.428, 5.899]
标签类别: tensor([0, 1, 2, 3])

测试多次增强的随机性:
  第1次 - 图像均值: -0.000, 标签类别: tensor([0, 1, 2, 3])
  第2次 - 图像均值: 0.000, 标签类别: tensor([0, 1, 2, 3])
  第3次 - 图像均值: 0.000, 标签类别: tensor([0, 1, 2, 3])


In [37]:
class Conv3DBlock(nn.Module):
    """一个简单的3D卷积块：Conv3D -> BatchNorm -> ReLU"""
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size, padding=padding),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

class DoubleConv(nn.Module):
    """标准的双卷积层：ConvBlock -> ConvBlock"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            Conv3DBlock(in_channels, out_channels),
            Conv3DBlock(out_channels, out_channels)
        )

    def forward(self, x):
        return self.double_conv(x)

class Encoder(nn.Module):
    """UNet的编码器部分，负责特征提取和下采样"""
    def __init__(self, in_channels=1, base_channels=32):
        super().__init__()
        self.levels = nn.ModuleList()

        # 定义每个级别的输入输出通道数
        self.channels = [in_channels, base_channels, base_channels*2, base_channels*4, base_channels*8]

        # 创建4个下采样级别
        for i in range(len(self.channels)-1):
            self.levels.append(
                nn.Sequential(
                    DoubleConv(self.channels[i], self.channels[i+1]),
                    nn.MaxPool3d(2)  # 下采样
                )
            )

    def forward(self, x):
        features = []  # 保存每个级别的特征，用于跳跃连接
        current_x = x

        for level in self.levels:
            # 通过双卷积
            conv_output = level[0](current_x)
            features.append(conv_output)  # 保存卷积后的特征（在下采样之前）
            # 下采样
            current_x = level[1](conv_output)

        return features, current_x

class Decoder(nn.Module):
    """UNet的解码器部分，负责上采样和特征融合"""
    def __init__(self, base_channels=32, num_classes=3):
        super().__init__()
        self.up_convs = nn.ModuleList()
        self.upsamples = nn.ModuleList()

        # 解码器的通道数序列（从深层到浅层）
        self.decoder_channels = [
            base_channels * 16,  # bottleneck输出
            base_channels * 8,   # 上采样后与编码器的base_channels*8拼接
            base_channels * 4,   # 上采样后与编码器的base_channels*4拼接
            base_channels * 2,   # 上采样后与编码器的base_channels*2拼接
        ]

        # 创建上采样路径
        for i in range(len(self.decoder_channels)-1):
            # 上采样层
            self.upsamples.append(
                nn.ConvTranspose3d(
                    self.decoder_channels[i],
                    self.decoder_channels[i] // 2,
                    kernel_size=2,
                    stride=2
                )
            )

            # 双卷积层：输入通道数 = 上采样输出 + 对应编码器特征的通道数
            in_ch = (self.decoder_channels[i] // 2) + (base_channels * 8 // (2 ** i))
            out_ch = self.decoder_channels[i+1]
            self.up_convs.append(DoubleConv(in_ch, out_ch))

        # 最终输出层
        self.final_conv = nn.Conv3d(base_channels * 2, num_classes, kernel_size=1)

    def forward(self, features, bottleneck):
        x = bottleneck

        # 反向遍历特征（从深层到浅层）
        for i, (upsample, up_conv) in enumerate(zip(self.upsamples, self.up_convs)):
            # 上采样
            x = upsample(x)

            # 获取对应的编码器特征（从后往前取）
            enc_feat = features[-(i+1)]

            # 调整特征尺寸（确保尺寸匹配）
            if x.shape != enc_feat.shape:
                x = center_crop_or_pad(x, enc_feat.shape[2:])

            # 跳跃连接：拼接编码器和解码器特征
            x = torch.cat([x, enc_feat], dim=1)
            x = up_conv(x)

        return self.final_conv(x)

def center_crop_or_pad(tensor, target_spatial_shape):
    """中心裁剪或填充张量到目标空间形状"""
    _, C, D, H, W = tensor.shape
    target_D, target_H, target_W = target_spatial_shape

    # 计算需要裁剪或填充的量
    diff_D = target_D - D
    diff_H = target_H - H
    diff_W = target_W - W

    if diff_D == 0 and diff_H == 0 and diff_W == 0:
        return tensor

    # 计算裁剪/填充的边界
    start_D = max(-diff_D // 2, 0)
    start_H = max(-diff_H // 2, 0)
    start_W = max(-diff_W // 2, 0)

    end_D = min(D, D - (diff_D + 1) // 2)
    end_H = min(H, H - (diff_H + 1) // 2)
    end_W = min(W, W - (diff_W + 1) // 2)

    # 先裁剪（如果需要）
    cropped = tensor[:, :, start_D:end_D, start_H:end_H, start_W:end_W]

    # 再填充（如果需要）
    if diff_D > 0 or diff_H > 0 or diff_W > 0:
        pad_D = (diff_D // 2, diff_D - diff_D // 2)
        pad_H = (diff_H // 2, diff_H - diff_H // 2)
        pad_W = (diff_W // 2, diff_W - diff_W // 2)
        cropped = F.pad(cropped, (pad_W[0], pad_W[1], pad_H[0], pad_H[1], pad_D[0], pad_D[1]))

    return cropped

In [38]:
class UNet3D(nn.Module):
    """完整的3D UNet模型"""
    def __init__(self, in_channels=1, num_classes=3, base_channels=32):
        super().__init__()
        self.encoder = Encoder(in_channels, base_channels)
        self.bottleneck = DoubleConv(base_channels*8, base_channels*16)
        self.decoder = Decoder(base_channels, num_classes)

    def forward(self, x):
        # 保存输入尺寸用于最终上采样
        input_size = x.shape[2:]

        # 编码路径
        features, bottleneck = self.encoder(x)

        # 桥接层
        bottleneck = self.bottleneck(bottleneck)

        # 解码路径
        output = self.decoder(features, bottleneck)

        # 如果输出尺寸不等于输入尺寸，进行上采样
        if output.shape[2:] != input_size:
            output = F.interpolate(output, size=input_size, mode='trilinear', align_corners=False)

        return output

In [28]:
def load_checkpoint_and_continue(model, optimizer, checkpoint_path):
    """加载检查点并继续训练"""
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_dice = checkpoint['best_val_dice']

    print(f"从epoch {start_epoch}恢复训练，最佳Dice: {best_val_dice:.4f}")
    return start_epoch, best_val_dice

In [30]:
def dice_coefficient(pred, target, smooth=1e-6):
    """计算Dice系数"""
    pred = torch.softmax(pred, dim=1)  # 将预测转换为概率 (B, C, D, H, W)

    # 将目标标签转换为one-hot编码 (B, D, H, W) -> (B, C, D, H, W)
    target_onehot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 4, 1, 2, 3).float()

    # 计算交集和并集
    intersection = (pred * target_onehot).sum(dim=(2, 3, 4))  # (B, C)
    union = pred.sum(dim=(2, 3, 4)) + target_onehot.sum(dim=(2, 3, 4))  # (B, C)

    # 计算每个类别的Dice系数
    dice = (2. * intersection + smooth) / (union + smooth)  # (B, C)

    return dice.mean(dim=0)  # 返回每个类别的平均Dice (C,)

In [33]:
import torch.nn as nn
import torch.nn.functional as F

class DiceCELoss(nn.Module):
    """Dice损失 + CrossEntropy损失的组合"""
    def __init__(self, weight_ce=0.5, weight_dice=0.5):
        super().__init__()
        self.weight_ce = weight_ce
        self.weight_dice = weight_dice
        self.ce = nn.CrossEntropyLoss()

    def forward(self, pred, target):
        # CrossEntropy损失
        ce_loss = self.ce(pred, target)

        # Dice损失 (1 - Dice系数)
        dice_loss = 1 - dice_coefficient(pred, target).mean()

        # 组合损失
        return self.weight_ce * ce_loss + self.weight_dice * dice_loss

In [53]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import time

# - optim.Adam: 自适应学习率优化器
# - ReduceLROnPlateau: 当验证损失停止改善时自动降低学习率

In [36]:
def create_brats_model():
    """创建适用于BraTS数据的4模态模型"""

    model = UNet3D(
        in_channels=4,      # 4个模态: t1, t1ce, t2, flair
        num_classes=4,      # 4个类别: 背景(0), 坏死(1), 水肿(2), 增强肿瘤(3)
        base_channels=32    # 基础通道数，控制模型大小
    )

    print("BraTS模型配置:")
    print(f"  输入通道: 4 (t1, t1ce, t2, flair)")
    print(f"  输出类别: 4 (背景, 坏死, 水肿, 增强肿瘤)")
    print(f"  基础通道: 32")
    print(f"  参数量: {sum(p.numel() for p in model.parameters()):,}")

    return model

In [29]:
# 创建数据加载器
print("创建数据加载器...")
train_loader, val_loader = create_brats_data_loaders()

# 检查数据加载器
print(f"训练集批次: {len(train_loader)}")
print(f"验证集批次: {len(val_loader)}")

创建数据加载器...
创建数据加载器...
总病例数: 369
训练集: 295 个病例
验证集: 74 个病例
✓ 数据加载器创建成功!
训练集批次: 295
验证集批次: 74


In [54]:
def main_training():
    """主训练函数"""
    # 设备配置
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")

    # 创建模型
    model = create_brats_model().to(device)

    # 定义优化器
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    # 不使用verbose，但手动记录学习率变化
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # 损失函数
    criterion = DiceCELoss(weight_ce=0.4, weight_dice=0.6)

    # 训练参数
    num_epochs = 20
    best_val_dice = 0.0
    checkpoint_dir = "/content/checkpoints"

    # 创建检查点目录
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"\n开始训练 {num_epochs} 个epochs...")
    print("=" * 60)

    # 记录训练过程
    train_losses, val_losses = [], []
    train_dices, val_dices = [], []

    start_time = time.time()

    # 记录初始学习率
    current_lr = optimizer.param_groups[0]['lr']
    print(f"初始学习率: {current_lr:.2e}")

    for epoch in range(num_epochs):
        epoch_start = time.time()
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        # === 训练阶段 ===
        model.train()
        train_loss = 0.0
        train_dice = 0.0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            dice = dice_coefficient(output, target).mean()

            train_loss += loss.item()
            train_dice += dice.item()

            if batch_idx % 20 == 0:
                print(f'  Batch {batch_idx:3d}/{len(train_loader)}, Loss: {loss.item():.4f}, Dice: {dice.item():.4f}')

        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_dice = train_dice / len(train_loader)
        train_losses.append(epoch_train_loss)
        train_dices.append(epoch_train_dice)

        # === 验证阶段 ===
        model.eval()
        val_loss = 0.0
        val_dice = 0.0

        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                dice = dice_coefficient(output, target).mean()

                val_loss += loss.item()
                val_dice += dice.item()

        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_dice = val_dice / len(val_loader)
        val_losses.append(epoch_val_loss)
        val_dices.append(epoch_val_dice)

        # 调整学习率前记录旧的学习率
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(epoch_val_loss)
        new_lr = optimizer.param_groups[0]['lr']

        # 手动打印学习率变化
        if new_lr < old_lr:
            print(f"Epoch {epoch+1}: 学习率从 {old_lr:.2e} 降低到 {new_lr:.2e}")

        epoch_time = time.time() - epoch_start

        print(f"训练 - Loss: {epoch_train_loss:.4f}, Dice: {epoch_train_dice:.4f}")
        print(f"验证 - Loss: {epoch_val_loss:.4f}, Dice: {epoch_val_dice:.4f}")
        print(f"时间: {epoch_time:.1f}s")

        # 保存最佳模型
        if epoch_val_dice > best_val_dice:
            best_val_dice = epoch_val_dice
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_dice': best_val_dice,
                'train_losses': train_losses,
                'val_losses': val_losses,
                'train_dices': train_dices,
                'val_dices': val_dices
            }
            torch.save(checkpoint, os.path.join(checkpoint_dir, 'best_model.pth'))
            print(f"✓ 保存最佳模型，验证Dice: {best_val_dice:.4f}")

        # 每10个epoch保存一次检查点
        if (epoch + 1) % 10 == 0:
            checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth')
            torch.save(checkpoint, checkpoint_path)
            print(f"✓ 保存检查点: {checkpoint_path}")

    total_time = time.time() - start_time
    print(f"\n训练完成! 总时间: {total_time/60:.1f}分钟")
    print(f"最佳验证Dice: {best_val_dice:.4f}")

    return model, train_losses, val_losses, train_dices, val_dices

In [56]:
# 首先创建数据加载器
print("创建数据加载器...")
train_loader, val_loader = create_brats_data_loaders()

# 开始训练！
print("\n" + "="*50)
print("开始BraTS模型训练")
print("="*50)

model, train_losses, val_losses, train_dices, val_dices = main_training()

创建数据加载器...
创建数据加载器...
总病例数: 369
训练集: 295 个病例
验证集: 74 个病例
✓ 数据加载器创建成功!

开始BraTS模型训练
使用设备: cuda
BraTS模型配置:
  输入通道: 4 (t1, t1ce, t2, flair)
  输出类别: 4 (背景, 坏死, 水肿, 增强肿瘤)
  基础通道: 32
  参数量: 22,484,484

开始训练 20 个epochs...
初始学习率: 1.00e-04

Epoch 1/20


KeyboardInterrupt: 